## 1. NLP Preprocessing Pipeline

### Task: Basic NLP Preprocessing

This task performs basic NLP preprocessing on the provided sentence. It includes word tokenization, English stopword removal, and stemming.


In [1]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Download English stopwords only if they are not already installed.
# The download needs internet access once; subsequent notebook runs reuse the corpus.
try:
    stopwords.words("english")
except LookupError:
    nltk.download("stopwords", quiet=True)

# Initialize these reusable objects ONCE, not every time the function runs.
english_stopwords = set(stopwords.words("english"))
stemmer = PorterStemmer()

In [2]:
def preprocess_sentence(sentence):
    # Step 1: Split the sentence into words and punctuation.
    tokens = re.findall(r"\w+|[^\w\s]", sentence)
    print("Original Tokens:", tokens)

    # Step 2: Remove English stopwords and punctuation.
    # Set membership is efficient and lower() makes comparisons case-insensitive.
    filtered_tokens = [
        token for token in tokens
        if token.isalpha() and token.lower() not in english_stopwords
    ]
    print("Tokens Without Stopwords:", filtered_tokens)

    # Step 3: Lowercase each remaining word and stem it.
    stemmed_words = [stemmer.stem(token.lower()) for token in filtered_tokens]
    print("Stemmed Words:", stemmed_words)

In [3]:
# Run Q1 using the EXACT sentence in the assignment.
sentence_q1 = "NLP techniques are used in virtual assistants like Alexa and Siri."
preprocess_sentence(sentence_q1)

Original Tokens: ['NLP', 'techniques', 'are', 'used', 'in', 'virtual', 'assistants', 'like', 'Alexa', 'and', 'Siri', '.']
Tokens Without Stopwords: ['NLP', 'techniques', 'used', 'virtual', 'assistants', 'like', 'Alexa', 'Siri']
Stemmed Words: ['nlp', 'techniqu', 'use', 'virtual', 'assist', 'like', 'alexa', 'siri']


### Observation

The tokenizer includes the period (`'.'`) in the original tokens. The second output excludes common stopwords such as `are`, `in`, and `and`, as well as punctuation. Porter stemming changes words such as `techniques` → `techniqu`, `assistants` → `assist`, and `used` → `use`. The word `used` remains in the filtered list because it is not included in NLTK's English stopword list.

## Short Answer Questions:

**1. What is the difference between stemming and lemmatization? Provide examples with the word “running.”**

**Stemming** removes word endings using simple rules and may produce words that are not valid dictionary words. **Lemmatization** uses vocabulary and grammatical information to return a word's dictionary base form.
For example, stemming changes "running" → "run", while lemmatization returns "run" when used as a verb but may retain "running" when used as a noun.

**2. Why might removing stop words be useful in some NLP tasks, and when might it actually be harmful?**
**Stopword removal is useful** in tasks such as keyword search and word-frequency analysis because common words like "the" and "of" often contribute little to the topic. Removing them reduces unnecessary features and helps the model focus on more meaningful words.

However, **stopword removal can be harmful** when meaning depends on grammar, context, or negation. For example, removing "not" from "not good" leaves "good," which may cause a negative statement to be incorrectly classified as positive in sentiment analysis.


## 2. Named Entity Recognition with spaCy

### Task: Named Entity Extraction

This task uses spaCy to identify named entities in the provided sentence and report each entity's text, label, and start and end character positions.


In [7]:
import spacy

# Load the pretrained model ONCE. It already contains a trained NER component.
nlp = spacy.load("en_core_web_sm")


def print_named_entities(sentence):
    # Process the sentence once using spaCy's NLP pipeline.
    doc = nlp(sentence)

    for entity in doc.ents:
        print(
            f"Text: {entity.text} | "
            f"Label: {entity.label_} | "
            f"Start: {entity.start_char} | "
            f"End: {entity.end_char}"
        )


# Keep the leading space because it affects character positions.
sentence_q2 = (
    " LeBron James as the NBA player of the United States in Laker "
    "and won the NBA Final MVP in 2020."
)

print_named_entities(sentence_q2)

Text: LeBron James | Label: PERSON | Start: 1 | End: 13
Text: NBA | Label: ORG | Start: 21 | End: 24
Text: the United States | Label: GPE | Start: 35 | End: 52
Text: Laker | Label: GPE | Start: 56 | End: 61
Text: MVP | Label: ORG | Start: 84 | End: 87
Text: 2020 | Label: DATE | Start: 91 | End: 95


### Observation

spaCy detects names, organizations, places, and dates and provides character positions in the original string. For example, `LeBron James` starts at index **1** because the sentence begins with a space. The model may occasionally assign an incorrect label (for example, the saved model output labels `Laker` as `GPE`). **The output is the model's prediction, not a manually corrected label.**


## Short Answer Questions:

**1. How does NER differ from POS tagging in NLP?**

**Named entity recognition (NER)** identifies named entities and their categories (for example, LeBron James → PERSON, 2020 → DATE). **Part-of-speech (POS) tagging** identifies the grammatical role of each word (for example, noun, verb, or adjective). NER asks “What named thing is this?” POS tagging asks “What grammatical role does this word play?”

**2. Describe two applications that use NER in the real world (e.g., financial news, search engines).**

**1. Healthcare Information Extraction:** NER can identify diseases, medications, and symptoms mentioned in medical records. For example, in “The patient was diagnosed with diabetes and prescribed metformin,” it can identify “diabetes” as a disease and “metformin” as a medication. Healthcare systems can use this information to organize patient records and support medical research.

**2. Legal Document Analysis:** NER can extract names of people, organizations, locations, and dates from legal documents. For example, in “Microsoft signed an agreement with ABC Company on June 15, 2025,” it can identify the organizations and agreement date. Legal professionals can use this information to review contracts, organize documents, and locate important details more efficiently.


## 3. Scaled Dot-Product Attention

### Task: Implement Scaled Dot-Product Attention

This task implements scaled dot-product attention using the given Query (Q), Key (K), and Value (V) matrices. The attention weights and final output matrix will be displayed when the code is run.


In [8]:
import numpy as np


def scaled_dot_product_attention(Q, K, V):
    # Step 1 and 2: Query-Key dot products, scaled by sqrt(Key dimension).
    key_dim = K.shape[1]
    scores = (Q @ K.T) / np.sqrt(key_dim)

    # Step 3: Numerically stable Softmax across each Query row.
    shifted_scores = scores - np.max(scores, axis=1, keepdims=True)
    exp_scores = np.exp(shifted_scores)
    attention_weights = exp_scores / np.sum(
        exp_scores, axis=1, keepdims=True
    )

    # Step 4: Use attention weights to combine the Value vectors.
    output = attention_weights @ V
    return attention_weights, output

In [9]:
# Use the exact Q, K, and V matrices supplied in Part II.
Q = np.array([[1, 0, 1, 0], [0, 1, 0, 1]])
K = np.array([[1, 0, 1, 0], [0, 1, 0, 1]])
V = np.array([[1, 2, 3, 4], [5, 6, 7, 8]])

attention_weights, output = scaled_dot_product_attention(Q, K, V)

print("Attention Weights:")
print(np.round(attention_weights, 3))
print("\nFinal Output:")
print(np.round(output, 3))

# Three quick checks: correct dimensions, normalized weights, known values.
assert attention_weights.shape == (2, 2) and output.shape == (2, 4)
np.testing.assert_allclose(attention_weights.sum(axis=1), [1, 1])
np.testing.assert_allclose(
    attention_weights,
    [[0.7310585786, 0.2689414214], [0.2689414214, 0.7310585786]],
    atol=1e-9
)

Attention Weights:
[[0.731 0.269]
 [0.269 0.731]]

Final Output:
[[2.076 3.076 4.076 5.076]
 [3.924 4.924 5.924 6.924]]


### Observation

The attention weights are approximately **`[[0.731, 0.269], [0.269, 0.731]]`**. Each row sums to 1. Query 1 gives more weight to Value 1, while Query 2 gives more weight to Value 2. The output is a **2 × 4** matrix because there are two Queries and each Value has four dimensions.

## Short Answer Questions:

**1. Why do we divide the attention score by √d in the scaled dot-product attention formula?**

When the Key/Query dimension is large, dot-product scores can become large and make Softmax highly concentrated, producing very small gradients. Dividing by √d keeps the score magnitude under better control and makes training more stable.

**2. How does self-attention help the model understand relationships between words in a sentence?**

Self-attention compares each token's Query with other tokens' Keys to calculate relevance, then combines their Values using attention weights. This allows each word to build a context-aware representation using information from other words, even when they are far apart. For example, in “I arrived at the bank after crossing the river,” the word “river” helps the model understand that “bank” refers to the side of a river.
